In [ ]:
import requests

In [52]:
COUCHDB_HOST = input("Ingrese el Host/IP de CouchDB: ").strip()
COUCHDB_PORT = input("Ingrese el Puerto de CouchDB: ").strip()
COUCHDB_USER = input("Ingrese el Usuario de CouchDB: ").strip()
COUCHDB_PASS = input("Ingrese la Contraseña de CouchDB: ").strip()
DATABASE_NAME = input("Ingrese el Nombre de la Base de Datos: ").strip()

# Construcción de URLs base

BASE_URL = f"http://{COUCHDB_HOST}:{COUCHDB_PORT}"
BULK_URL = f"{BASE_URL}/{DATABASE_NAME}/_bulk_docs"
AUTH = (COUCHDB_USER, COUCHDB_PASS)

# Creacion de indices

In [53]:
def crear_indice_1 ():
    url = BASE_URL + f"/{DATABASE_NAME}/_index"
    payload_idx1 = {
        "index": {"fields": ["tipo_doc", "categoria", "precio"]},
        "name": "idx_categoria_precio",
        "type": "json"
    }
    res = requests.post(url, json=payload_idx1, auth=AUTH)
    print("Respuesta creación índice:", res.json())

In [62]:
def crear_indice_2():
    url = BASE_URL + f"/{DATABASE_NAME}/_index"
    payload_idx2 = {
        "index": {"fields": ["tipo_doc", "cliente_id"]},
        "name": "idx_pedidos_cliente",
        "type": "json"
    }
    res = requests.post(url, json=payload_idx2, auth=AUTH)
    print("Respuesta creación índice:", res.json())

In [63]:
def crear_indice_3():
    url = BASE_URL + f"/{DATABASE_NAME}/_index"
    payload_idx3 = {
        "index": {"fields": ["tipo_doc", "categoria", "atributos.garantia_meses"]},
        "name": "idx_atributo_especifico",
        "type": "json"
    }
    res = requests.post(url, json=payload_idx3, auth=AUTH)
    print("Respuesta creación índice:", res.json())

# Búsqueda combinando categoría, rango de precio y atributo específico

In [56]:
def buscar_por_atributo_especifico():
    url = BASE_URL + f"/{DATABASE_NAME}/_find"
    payload = {
        "selector": {
            "tipo_doc": "producto",
            "categoria": "Electrónica",
            "precio": {"$gte": 50, "$lte": 500},
            "atributos.garantia_meses": 12
        },
        "execution_stats": True
    }
    res = requests.post(url, json=payload, auth=AUTH).json()
    print(f"Encontrados {len(res.get('docs', []))} productos con garantía de 12 meses.")

# Recuperar pedidos de un cliente y actualizar el estado de uno

In [57]:
def recuperar_y_actualizar_pedido(cliente_id="CLI-000001"):
    url_find = BASE_URL + f"/{DATABASE_NAME}/_find"
    
    # 1. Recuperar pedidos del cliente
    payload = {
        "selector": {
            "tipo_doc": "pedido",
            "cliente_id": cliente_id
        }
    }
    res = requests.post(url_find, json=payload, auth=AUTH).json()
    pedidos = res.get("docs", [])
    print(f"Se recuperaron {len(pedidos)} pedidos para el cliente {cliente_id}.")

# Dos consultas de agregación utilizando MapReduce de CouchDB

In [58]:
def crear_vistas_agregacion():
    url_design = BASE_URL + f"/{DATABASE_NAME}/_design/agregaciones"
    
    # Documento de diseño con MapReduce
    
    ddoc = {
        "_id": "_design/agregaciones",
        "views": {
            # Agregación 1: Total de ventas globales / stats de montos de pedidos
            "monto_total_ventas": {
                "map": "function (doc) { if (doc.tipo_doc === 'pedido' && doc.monto_total) { emit(doc.estado, doc.monto_total); } }",
                "reduce": "_stats"
            },
            # Agregación 2: Conteo de productos por categoría
            "conteo_por_categoria": {
                "map": "function (doc) { if (doc.tipo_doc === 'producto' && doc.categoria) { emit(doc.categoria, 1); } }",
                "reduce": "_count"
            }
        }
    }
    # Guardar documento de diseño
    
    requests.put(url_design, json=ddoc, auth=AUTH)
    
    # Consultar Agregación 1 (Ventas por estado)
    
    url_query1 = f"{url_design}/_view/monto_total_ventas?group=true"
    res1 = requests.get(url_query1, auth=AUTH).json()
    
    print("Agregación ejecutada (Métricas/Stats por estado de pedido):")
    print(res1.get("rows", []))

# Prueba requitico 3 y 4

In [59]:
def ejecutar_consulta():
    # Cambiado _index por _find
    url = BASE_URL + f"/{DATABASE_NAME}/_find" 
    payload = {
        "selector": {
            "tipo_doc": "producto",
            "categoria": "Electrónica",
            "precio": {"$gte": 100, "$lte": 300}
        },
        "execution_stats": True  # Activa las métricas de rendimiento en CouchDB
    }
    res = requests.post(url, json=payload, auth=AUTH)
    data = res.json()
    
    print("Resultados encontrados:", len(data.get('docs', [])))
    print("Métricas de ejecución:", data.get('execution_stats'))

# Demostrar evolución del esquema agregando nueva categoría sin migración relacional
## Prueba 5 a 7

In [60]:
def demostrar_evolucion_esquema():
    nuevo_producto = {
        "tipo_doc": "producto",
        "producto_id": "PROD-999999",
        "nombre": "Drone de Monitoreo Agrícola",
        "categoria": "Tecnología Agrícola",  # Categoria totalmente nueva
        "precio": 2500.00,
        "stock": 10,
        "atributos": {
            "autonomia_minutos": 45,
            "camara_multiespectral": True,
            "alcance_km": 12.5
        }
    }
    url = BASE_URL + f"/{DATABASE_NAME}"
    res = requests.post(url, json=nuevo_producto, auth=AUTH).json()
    print(f"[OK] Requisito 7: Documento con esquema evolucionado insertado correctamente (ID: {res.get('id')}).")

# Corrida de funciones

In [64]:
if __name__ == "__main__":
    print("=== EXECUTANDO PRUEBAS Y REQUISITOS MERCADOTICO 360 ===")

    # 1. Crear índices (Requisito 3)
    crear_indice_1()
    crear_indice_2()
    crear_indice_3()

    # 2. Consultas de búsqueda (Requisito 4)
    
    print("\n--- Ejecutando búsquedas filtradas ---")
    ejecutar_consulta()  # Tu consulta existente por categoría y precio
    buscar_por_atributo_especifico()  # Búsqueda con atributo heterogéneo anidado

    # 3. Operaciones sobre clientes y pedidos (Requisito 5)
    
    print("\n--- Consultando y actualizando pedidos ---")
    recuperar_y_actualizar_pedido("CLI-000001")

    # 4. Agregaciones MapReduce (Requisito 6)
    
    print("\n--- Ejecutando vistas MapReduce ---")
    crear_vistas_agregacion()

    # 5. Evolución del esquema (Requisito 7)
    
    print("\n--- Demostrando evolución del esquema ---")
    demostrar_evolucion_esquema()

=== EXECUTANDO PRUEBAS Y REQUISITOS MERCADOTICO 360 ===
Respuesta creación índice: {'result': 'exists', 'id': '_design/050d7553f13d10d34aa300d7d5f61bcaa425d2c6', 'name': 'idx_categoria_precio'}
Respuesta creación índice: {'result': 'created', 'id': '_design/903576b7732e961c508013f01b0da6c9de595163', 'name': 'idx_pedidos_cliente'}
Respuesta creación índice: {'result': 'created', 'id': '_design/57829f0bd0712da14f74c1a89eec6dc0f495c2df', 'name': 'idx_atributo_especifico'}

--- Ejecutando búsquedas filtradas ---
Resultados encontrados: 25
Métricas de ejecución: {'total_keys_examined': 26, 'total_docs_examined': 26, 'total_quorum_docs_examined': 0, 'results_returned': 25, 'execution_time_ms': 19.0}
Encontrados 25 productos con garantía de 12 meses.

--- Consultando y actualizando pedidos ---
Se recuperaron 9 pedidos para el cliente CLI-000001.

--- Ejecutando vistas MapReduce ---
Agregación ejecutada (Métricas/Stats por estado de pedido):
[{'key': 'cancelado', 'value': {'sum': 124477711.070